## Aeropulse — Orchestration: 00 Create Control Table

**Purpose:** One-off (but safe to re-run) setup — creates the `control` schema and `batch_control` table, and seeds it with every expected `flight` batch as `PENDING`. `01 identify next batch` reads from this table to drive the orchestration loop.

**Not pipeline-parameterised** — run manually, or once from the pipeline before the batch loop starts.


In [1]:
from pyspark.sql import functions as F

StatementMeta(, 61bff56c-0b55-4211-b7eb-d4098896d20d, 3, Finished, Available, Finished, False)

In [2]:
# create the 'control' schema in the attached lakehouse (aeropulse_control_lh) if it doesn't already exist
spark.sql("CREATE SCHEMA IF NOT EXISTS control")

# create the batch_control table if it doesn't already exist — one row per (source_name, batch_id)
# status moves PENDING -> IN_PROGRESS -> COMPLETED, or -> FAILED on error (see 02/03/04 notebooks)
spark.sql("""
CREATE TABLE IF NOT EXISTS control.batch_control (
    source_name STRING,        -- which source this batch belongs to, e.g. 'flight'
    batch_id STRING,           -- e.g. '2018_01' — the unit of work the pipeline processes
    batch_year STRING,         -- the year portion of batch_id, kept separate for convenience/filtering
    status STRING,             -- PENDING | IN_PROGRESS | COMPLETED | FAILED
    start_time TIMESTAMP,      -- set when the batch moves to IN_PROGRESS
    end_time TIMESTAMP,        -- set when the batch moves to COMPLETED or FAILED
    error_message STRING,      -- populated by 04-fail-batch when a run fails
    retry_count INT,           -- incremented each time a batch fails, for visibility/alerting
    created_timestamp TIMESTAMP,
    updated_timestamp TIMESTAMP
)
USING DELTA
""")


StatementMeta(, 61bff56c-0b55-4211-b7eb-d4098896d20d, 4, Finished, Available, Finished, False)

DataFrame[]

In [3]:
# build the full list of expected batches: one per month from 2018-01 through 2026-12
# NOTE: this only seeds batch_ids — it doesn't check that a source file actually exists in ADLS for
# each one yet. If any of the later months don't have a source file behind them, the pipeline will
# pick them up as "next batch" and fail on the landing step. Trim the end of the range, or switch to
# discovering batches from the actual ADLS folder listing, if that turns out to be the case.
expected_batches = [
    f"{year}_{month:02d}"
    for year in range(2018, 2027)   # FIX: was range(2018, 2017) — start > stop, so this always produced an empty list
    for month in range(1, 13)
]

# FIX: batch_year is now derived from each batch_id instead of being hardcoded to "2018" —
# the previous version stamped every batch (including 2019-2026 ones) with batch_year = "2018"
seed_df = (
    spark.createDataFrame(
        [("flight", b, b.split("_")[0]) for b in expected_batches],
        ["source_name", "batch_id", "batch_year"]
    )
    .withColumn("status", F.lit("PENDING"))
    .withColumn("start_time", F.lit(None).cast("timestamp"))
    .withColumn("end_time", F.lit(None).cast("timestamp"))
    .withColumn("error_message", F.lit(None).cast("string"))
    .withColumn("retry_count", F.lit(0))
    .withColumn("created_timestamp", F.current_timestamp())
    .withColumn("updated_timestamp", F.current_timestamp())
)

# only insert batches not already tracked, so re-running this notebook is safe
# (won't duplicate rows or reset the status of batches already in progress/completed)
if spark.catalog.tableExists("control.batch_control"):
    existing = spark.sql("SELECT source_name, batch_id FROM control.batch_control")
    seed_df = seed_df.join(existing, ["source_name", "batch_id"], "left_anti")

seed_df.write.format("delta").mode("append").saveAsTable("control.batch_control")


StatementMeta(, 61bff56c-0b55-4211-b7eb-d4098896d20d, 5, Finished, Available, Finished, False)